# Results Presentation: Tables, Figures, and Error Bars

## Overview
How results are presented is as important as the analysis itself. Poor presentation obscures findings; good presentation makes them reproducible and unambiguous.

**Key decisions covered here:**

1. **Error bars:** SD vs SE vs CI — which to use and when
2. **ANOVA and regression tables:** what to report
3. **Figure design:** axis labelling, colour, raw data
4. **Reporting mixed models and GLMs:** random effects, overdispersion

**Reference:** Quinn & Keough (2002) ch. 19; Cumming et al. (2007) Psychological Science; Weissgerber et al. (2015) PLOS Biology

---

In [ ]:
library(tidyverse); library(lme4); library(lmerTest); library(emmeans)
set.seed(12)
# Simulated plant height data: 3 nutrient treatments, n=15 per group
dat <- data.frame(
  treatment = rep(c("control", "low_N", "high_N"), each = 15),
  height    = c(rnorm(15, 22, 4), rnorm(15, 28, 4), rnorm(15, 35, 5))
) |> mutate(treatment = factor(treatment, levels = c("control","low_N","high_N")))

# Summary statistics
summary_stats <- dat |>
  group_by(treatment) |>
  summarise(
    n    = n(),
    mean = mean(height),
    sd   = sd(height),
    se   = sd(height) / sqrt(n()),
    ci95 = qt(0.975, n()-1) * se,
    .groups = "drop"
  )
cat("Summary statistics:\n")
print(summary_stats |> mutate(across(mean:ci95, \(x) round(x, 2))))

---
## Error bars: SD vs SE vs 95% CI

In [ ]:
cat("=== When to use each error measure ===\n")
cat("\nSD (Standard Deviation):\n")
cat("  - Describes the spread of the DATA (biological variability)\n")
cat("  - Does NOT shrink with n\n")
cat("  - Use when: describing the distribution of raw values,\n")
cat("    reporting normal ranges, communicating biological variation\n")

cat("\nSE (Standard Error of the mean):\n")
cat("  - Describes precision of the ESTIMATED MEAN\n")
cat("  - SE = SD / sqrt(n): shrinks with n\n")
cat("  - Avoid as error bars: SE bars are often misread as CIs\n")
cat("  - Use when: standard format in some fields (e.g. physiology),\n")
cat("    but always label explicitly as SE\n")

cat("\n95% CI (Confidence Interval):\n")
cat("  - If this procedure were repeated, 95% of such intervals contain the true mean\n")
cat("  - Non-overlapping CIs suggest (but do not confirm) significance at α≈0.05\n")
cat("  - PREFERRED for most presentations: directly interpretable,\n")
cat("    aligned with hypothesis testing, least ambiguous\n")

# Visualise all three
plot_data <- bind_rows(
  summary_stats |> mutate(error = sd, type = "Mean ± SD"),
  summary_stats |> mutate(error = se, type = "Mean ± SE"),
  summary_stats |> mutate(error = ci95, type = "Mean ± 95% CI")
)
ggplot(plot_data, aes(treatment, mean, colour = type)) +
  geom_point(size = 3, position = position_dodge(0.5)) +
  geom_errorbar(aes(ymin = mean - error, ymax = mean + error),
                width = 0.2, linewidth = 0.8,
                position = position_dodge(0.5)) +
  facet_wrap(~ type, scales = "free_y") +
  labs(title = "SD vs SE vs 95% CI error bars (same data)",
       y = "Plant height (cm)", x = NULL) +
  theme_minimal() + theme(legend.position = "none")

---
## Publication-quality figures with raw data

In [ ]:
# Best practice: show raw data alongside summary statistics
ggplot(summary_stats, aes(treatment, mean)) +
  # Raw data as jittered points (behind summaries)
  geom_jitter(data = dat, aes(y = height), width = 0.1, alpha = 0.4,
              colour = "grey50", size = 1.8) +
  # 95% CI error bar
  geom_errorbar(aes(ymin = mean - ci95, ymax = mean + ci95),
                width = 0.15, linewidth = 1.0, colour = "#2166ac") +
  # Mean as point (on top of raw data)
  geom_point(size = 4, colour = "#2166ac") +
  labs(title = "Plant height by nutrient treatment",
       subtitle = "Points = raw data; error bars = mean ± 95% CI; n = 15 per group",
       y = "Plant height (cm)",
       x = "Nutrient treatment") +
  scale_x_discrete(labels = c("Control", "Low N", "High N")) +
  theme_minimal(base_size = 13)

cat("Key figure principles:\n")
cat("  - Show raw data points wherever feasible (Weissgerber et al. 2015)\n")
cat("  - Label error bars explicitly in figure caption\n")
cat("  - Avoid dual y-axes (they obscure relationships)\n")
cat("  - Use greyscale-safe colour palettes (colourblind accessibility)\n")
cat("  - Include n in caption or on plot\n")

---
## Reporting ANOVA and regression tables

In [ ]:
# ANOVA table: what to report
aov_fit <- aov(height ~ treatment, data = dat)
cat("=== One-way ANOVA table ===\n")
print(summary(aov_fit))
cat("\nReport: F(df_treatment, df_residual) = X.XX, p = .XXX, eta² = XX%\n")
ss <- summary(aov_fit)[[1]][["Sum Sq"]]
eta2 <- ss[1] / sum(ss)
cat(sprintf("Example: F(2, 42) = %.2f, p = %.4f, eta² = %.2f\n",
            summary(aov_fit)[[1]]$`F value`[1],
            summary(aov_fit)[[1]]$`Pr(>F)`[1],
            eta2))

# Post-hoc comparisons via emmeans
cat("\n=== Post-hoc comparisons (Tukey HSD) ===\n")
em <- emmeans(aov_fit, ~ treatment)
contrasts <- pairs(em, adjust = "tukey")
print(contrasts)
cat("\nReport: include all pairwise differences with 95% CI and adjusted p-values\n")

In [ ]:
# Regression table: what to report
reg_dat <- data.frame(
  x = rnorm(50, 10, 3),
  y = 2 + 1.5 * rnorm(50, 10, 3) + rnorm(50, 0, 4)
)
lm_fit <- lm(y ~ x, data = reg_dat)
cat("=== Linear regression: what to report ===\n")
print(summary(lm_fit))
cat("\nMinimum to report:\n")
cat("  - Slope estimate with 95% CI (not just SE)\n")
cat("  - Standardised beta (scale-independent effect size)\n")
cat("  - R² (overall model fit)\n")
cat("  - Residual standard error (prediction uncertainty)\n")
cat("  - F-statistic and p-value for overall model\n")

# Mixed model reporting
cat("\n=== Mixed model: what to report ===\n")
cat("  Fixed effects: estimate, SE, t/z value, df (Satterthwaite), p\n")
cat("  Random effects: variance (σ²) and SD for each grouping factor\n")
cat("  ICC: proportion of variance at grouping level\n")
cat("  Number of groups and observations at each level\n")
cat("  Model comparison: LRT or AIC for fixed effects selection\n")
cat("  Never report only p-values — include effect sizes with CIs\n")

---
## Common Pitfalls

**1. Using SE error bars without labelling them**
SE bars are much narrower than 95% CI bars and are frequently mistaken for CIs by readers. If you use SE, label them explicitly (e.g., "±1 SE") in the figure caption and on the figure. Better still, use 95% CI which is directly interpretable.

**2. Interpreting overlapping CI bars as non-significance**
Two 95% CI bars can overlap while the pairwise comparison is still significant at α = 0.05. Overlapping CIs indicate approximate significance at α ≈ 0.01 for two independent means. Do not substitute visual inspection of error bars for a formal test.

**3. Reporting p-values without effect sizes**
A statistically significant effect with p = 0.001 can be biologically trivial if n is large. Always report an effect size (Cohen's d, eta², partial eta², R², etc.) alongside p-values. CIs on effect sizes are even better.

**4. Bar plots that hide the data distribution**
Bar-and-whisker plots show only mean and error — they can look identical for normally distributed and heavily skewed data. Use violin plots, boxplots with overlaid points, or strip charts (jitter + summary) to show the full distribution, especially for small-to-moderate n.

**5. Using too many decimal places**
Reporting p = 0.0423871 implies false precision. Report p to 3 decimal places (p = .042) or use thresholds (p < .05). For p < .001, report as "p < .001" not "p = 0.0000023". Match decimal places to measurement precision.

**6. Not reporting random effects in mixed models**
Omitting variance components from mixed model write-ups makes results non-reproducible. Report the SD (or variance) for each random effect term, and the ICC. Readers need this to assess the clustering structure and to re-use the estimates.


---
*r_methods_library - Samantha McGarrigle*